# Module 3 - Dynamic MFA EU

### Group 57

### Step 1: Importing your packages 

In [133]:
import pandas as pd
import numpy as np
import plotly.express as px
from flodym.export import PlotlySankeyPlotter
from plotly.colors import qualitative
import plotly.graph_objects as go
import matplotlib.pyplot as plt

In [134]:
from flodym import Dimension, DimensionSet, FlodymArray,  FlowDefinition, MFASystem , Parameter, InflowDrivenDSM, NormalLifetime, StockDefinition, SimpleFlowDrivenStock
from flodym import make_processes, make_empty_flows, make_empty_stocks
from flodym.export import PlotlySankeyPlotter

### Step 2: Importing Flodym package which we will use to set up dynamic MFA

If you have not run FLODYM before, you will need to run the installer below.

In [ ]:
# !pip install flodym

### Step 3: Setting up our Dimensions/DimensionSets 

We define our systems dimensions for time as years, and products by material types. In this model we will have a region, time, and materials.


In [135]:
# Define time dimension
time_dim = Dimension(
    name = "time", 
    letter = "t", 
    items = list(range(2010, 2061)),
    unit = "year"
)

#Define products dimension
product_dim = Dimension(
    name="Products",
    letter="p", 
    dtype=str, 
    items=["Biomass", "Metal ores (gross ores)", "Non-metallic minerals", "Fossil energy materials_carrier"], #Material types
    unit="Kt"
    )

# Now we can combine these dimensions into a dimensionset
dims = DimensionSet(dim_list=[time_dim, product_dim])


### Step 4: Setting up our processes

Here we define the processes of our system, add them to a list and give them the dimensions of our system (time, products).

Note that the procesz name in position 0 must be named "sysenv". This is a requirement from the FLODYM package and simply allows us to import and export mass in and out of our system boundaries.

In [136]:
process_names = [
    "sysenv",
    "Natural resources extracted",
    "Imports",
    "Direct material inputs",
    "Processed material",
    "Exports",
    "Dissipative flows",
    "Total emissions",
    "Emissions to air",
    "Emissions to water",
    "Material use",
    "Material accumulation",
    "Waste treatment",
    "Incineration",
    "Waste landfilled",
    "Recycling",
    "Backfilling",
    "Residual imbalance"
]

processes = make_processes(process_names)

print(list(processes))
print(f"process count: {len(processes)}")
dims[("t", "p")].shape

['sysenv', 'Natural resources extracted', 'Imports', 'Direct material inputs', 'Processed material', 'Exports', 'Dissipative flows', 'Total emissions', 'Emissions to air', 'Emissions to water', 'Material use', 'Material accumulation', 'Waste treatment', 'Incineration', 'Waste landfilled', 'Recycling', 'Backfilling', 'Residual imbalance']
process count: 18


(51, 4)

In [137]:
print("Processes:")
print(list(processes))
print("Process count:", len(processes))

print("\nDimension sizes:")
print("time:", len(dims["t"].items))
print("products:", len(dims["p"].items))

print("\nShapes:")
print("(t,p):", (len(dims["t"].items), len(dims["p"].items)))


Processes:
['sysenv', 'Natural resources extracted', 'Imports', 'Direct material inputs', 'Processed material', 'Exports', 'Dissipative flows', 'Total emissions', 'Emissions to air', 'Emissions to water', 'Material use', 'Material accumulation', 'Waste treatment', 'Incineration', 'Waste landfilled', 'Recycling', 'Backfilling', 'Residual imbalance']
Process count: 18

Dimension sizes:
time: 51
products: 4

Shapes:
(t,p): (51, 4)


### Step 5: Setting up flows

Each flow is named by its source- and target process from the defined process names above, and defined in a dictionary. 

To comply with mass balance, which we will check for later, we will need to add flows for imports and exports of our system. 

This means we must define the flows $sysenv \rightarrow imports$ and $Exports \rightarrow sysenv$.

We then define all flows within our system.

Lastly, to comply with mass balance despite possible inaccuracies in our raw data, we define "Residual imbalance" flows where mass that is not correctly attained for in our raw data is aggregated. This ensures our system will be in mass balance.

In [138]:
flow_definitions = [

    # SYSTEM IMPORTS
    FlowDefinition(
        from_process_name="sysenv",
        to_process_name="Imports",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="sysenv",
        to_process_name="Natural resources extracted",
        dim_letters=("t", "p")
    ),

    # DIRECT MATERIAL INPUTS
    FlowDefinition(
        from_process_name="Imports",
        to_process_name="Direct material inputs",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Natural resources extracted",
        to_process_name="Direct material inputs",
        dim_letters=("t", "p")
    ),

    # PROCESSED MATERIAL
    FlowDefinition(
        from_process_name="Direct material inputs",
        to_process_name="Processed material",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Processed material",
        to_process_name="Exports",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Processed material",
        to_process_name="Dissipative flows",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Processed material",
        to_process_name="Total emissions",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Processed material",
        to_process_name="Material use",
        dim_letters=("t", "p")
    ),

    # TOTAL EMISSIONS
    FlowDefinition(
        from_process_name="Total emissions",
        to_process_name="Emissions to air",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Total emissions",
        to_process_name="Emissions to water",
        dim_letters=("t", "p")
    ),

    # MATERIAL USE
    FlowDefinition(
        from_process_name="Material use",
        to_process_name="Waste treatment",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Material use",
        to_process_name="Material accumulation",
        dim_letters=("t", "p")
    ),

    # WASTE TREATMENT
    FlowDefinition(
        from_process_name="Waste treatment",
        to_process_name="Incineration",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Incineration",
        to_process_name="Total emissions",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Waste treatment",
        to_process_name="Waste landfilled",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Waste treatment",
        to_process_name="Backfilling",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Backfilling",
        to_process_name="Processed material",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Waste treatment",
        to_process_name="Recycling",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Recycling",
        to_process_name="Processed material",
        dim_letters=("t", "p")
    ),

    # RESIDUAL MASS-BALANCING FLOWS
    FlowDefinition(
        from_process_name="Direct material inputs",
        to_process_name="Residual imbalance",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Processed material",
        to_process_name="Residual imbalance",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Material use",
        to_process_name="Residual imbalance",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Total emissions",
        to_process_name="Residual imbalance",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Waste treatment",
        to_process_name="Residual imbalance",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Recycling",
        to_process_name="Residual imbalance",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Backfilling",
        to_process_name="Residual imbalance",
        dim_letters=("t", "p")
    ),
    FlowDefinition(
        from_process_name="Incineration",
        to_process_name="Residual imbalance",
        dim_letters=("t", "p")
    ),

    # EXPORT FLOWS
    FlowDefinition(
        from_process_name="Exports", 
        to_process_name="sysenv", 
        dim_letters=("t","p",)
    ),
    FlowDefinition(
        from_process_name="Dissipative flows", 
        to_process_name="sysenv", 
        dim_letters=("t","p",)
    ),
    FlowDefinition(
        from_process_name="Waste landfilled", 
        to_process_name="sysenv", 
        dim_letters=("t","p",)
    ),
    FlowDefinition(
        from_process_name="Emissions to air", 
        to_process_name="sysenv", 
        dim_letters=("t","p",)
    ),
    FlowDefinition(
        from_process_name="Emissions to water", 
        to_process_name="sysenv", 
        dim_letters=("t","p",)
    ),
    
    # tHE TOTAL RESIDUAL MASS FLOW
    FlowDefinition(
        from_process_name="Residual imbalance",
        to_process_name="sysenv",
        dim_letters=("t", "p")
    )
]


flows = make_empty_flows(processes=processes, flow_definitions=flow_definitions, dims=dims)

#check the names of your flows
print("Flow names:\n ", "\n  ".join(flows))
# check the number of flows in your system
print (f"flow count: {len(flows)}")
# check your matrix dimensions
dims[("t","p")].shape

Flow names:
  sysenv => Imports
  sysenv => Natural resources extracted
  Imports => Direct material inputs
  Natural resources extracted => Direct material inputs
  Direct material inputs => Processed material
  Processed material => Exports
  Processed material => Dissipative flows
  Processed material => Total emissions
  Processed material => Material use
  Total emissions => Emissions to air
  Total emissions => Emissions to water
  Material use => Waste treatment
  Material use => Material accumulation
  Waste treatment => Incineration
  Incineration => Total emissions
  Waste treatment => Waste landfilled
  Waste treatment => Backfilling
  Backfilling => Processed material
  Waste treatment => Recycling
  Recycling => Processed material
  Direct material inputs => Residual imbalance
  Processed material => Residual imbalance
  Material use => Residual imbalance
  Total emissions => Residual imbalance
  Waste treatment => Residual imbalance
  Recycling => Residual imbalance
  Bac

(51, 4)

### Step 6: Setting up Parameters

To set up parameters we need to import our raw data. To do this we firstly donwload the excel sheet provided in the same folder as this notebook. 

**Make sure to place the Jupyter notebooks and excel sheets in the same folder.**

We will then read through the excel file and loop through the individual sheets for "Biomass", "Metal ores (gross ores)", "Non-metallic minerals", "Fossil energy materials_carrier" so we get the data for each year and each material category. 

With this we set up our parameter objects. You mat open the excel file to see that our parameters are simply named after the first row in each column. If you have opened the excel, remember to close it again before running continuing to running the notebook, otherwise your code will be denied permission to read the file.

In [139]:
import numpy as np
import pandas as pd

workbook_path = "f1_f17_all_materials.xlsx"
sheets = pd.read_excel(workbook_path, sheet_name=None)

# Extract product names from sheet names
products = list(sheets.keys())
print(products)

# Define parameters you care about
desired_parameters = [
    'nre', 'imports', 'pm', 'exports', 'dis flow', 'tem', 'matuse',
    'mu2wt', 'mu2ma', 'wt2te', 'te2air', 'te2water', 'wt2land',
    'wt2rec', 'wt2back', 'rec2pm', 'back2pm'
]

# Storage for parameter arrays
parameter_data = {name: [] for name in desired_parameters}

# First pass: determine the *maximum* length across all sheets
max_length = 0
for product in products:
    df = sheets[product]

    # Keep only columns that have a real header
    df = df.loc[:, df.columns.notna() & (df.columns != "")]

    max_length = max(max_length, len(df))

print("Max length across all products =", max_length)
# Second pass: load values and pad to max_length
for product in products:
    df = sheets[product]

    # Keep only columns with valid header
    df = df.loc[:, df.columns.notna() & (df.columns != "")]

    current_length = len(df)

    for param_name in desired_parameters:
        if param_name in df.columns:
            col = df[param_name].values.reshape(-1, 1)
        else:
            col = np.zeros((current_length, 1))

        # Pad with zeros if needed
        if current_length < max_length:
            padding = np.zeros((max_length - current_length, 1))
            col = np.vstack([col, padding])

        parameter_data[param_name].append(col)

# Create Parameter objects safely
parameters = {
    name: Parameter(
        name=name,
        dims=dims[("t", "p")],
        values=np.hstack(values_list)
    )
    for name, values_list in parameter_data.items()
}

# Print results
for name, param in parameters.items():
    print(f"{name}: shape={param.values.shape}")


['Biomass', 'Metal ores (gross ores)', 'Non-metallic minerals', 'Fossil energy materials_carrier']
Max length across all products = 51
nre: shape=(51, 4)
imports: shape=(51, 4)
pm: shape=(51, 4)
exports: shape=(51, 4)
dis flow: shape=(51, 4)
tem: shape=(51, 4)
matuse: shape=(51, 4)
mu2wt: shape=(51, 4)
mu2ma: shape=(51, 4)
wt2te: shape=(51, 4)
te2air: shape=(51, 4)
te2water: shape=(51, 4)
wt2land: shape=(51, 4)
wt2rec: shape=(51, 4)
wt2back: shape=(51, 4)
rec2pm: shape=(51, 4)
back2pm: shape=(51, 4)


### Step 7: Setting up the Stocks

We define a stock in our system for the models "in use" stock. From our modelled system we see that the only stock is "Material Accumulation". 

We define this stock with the flow going into our "Material Accumulation" process.

In [140]:
stock_defs = [
    StockDefinition(
        dim_letters=("t", "p"),
        name="in use",
        subclass=SimpleFlowDrivenStock,
        inflow= flows["Material use => Material accumulation"],
        process_name="Material accumulation",
        time_letter="t"
    )
]

stocks = make_empty_stocks(stock_definitions=stock_defs, processes=processes, dims=dims)
print(list(stocks.keys()))

['in use']


### Step 8: Setting up calculations.

For our dynamic MFA we set up equations for our flows based on our parameters defined above, so we can impliment time into our model and utilize our historic data.

For our dynamic MFA we set up the equations of our flows based on the paramters we defined above. This will allow us to implement time and historic data into our model.

Each flow equation is defined as self.flows["process where mass leaves => process where mass goes"].

FLODYM calculates mass balance by checking if our system's imports and exports are equal.

We will se later that this is not true in our case due to stock accumulation. This will be explained further later in the notebook.

For now, we need the import and export values to be defined by the sysenv flows. With flows going **from** sysenv being imports, and flows going **to** sysenv being exports.

To calculate for mass balance, we define the total inflow- and outflow values for each process and add the difference between these to our residual imbalance, for each process.

In [141]:
class MyMFASystem(MFASystem):

    def compute(self):

        p = self.parameters
        f = self.flows

        #IMPORTS
        f["sysenv => Imports"][...] = p["imports"]
        f["sysenv => Natural resources extracted"][...] = p["nre"]

        #FLOWS WITHIN SYSTEM BOUNDARIES
        f["Imports => Direct material inputs"][...] = p["imports"]

        f["Natural resources extracted => Direct material inputs"][...] = p["nre"]

        f["Direct material inputs => Processed material"][...] = p["pm"] - p["rec2pm"] - p["back2pm"]
        
        f["Processed material => Exports"][...] = p["exports"]
        f["Processed material => Dissipative flows"][...] = p["dis flow"]
        f["Processed material => Total emissions"][...] = p["tem"] - p["wt2te"]
        f["Processed material => Material use"][...] = p["matuse"]

        f["Total emissions => Emissions to air"][...] = p["te2air"]
        f["Total emissions => Emissions to water"][...] = p["te2water"]

        f["Material use => Waste treatment"][...] = p["mu2wt"]
        f["Material use => Material accumulation"][...] = p["mu2ma"]

        f["Waste treatment => Incineration"][...] = p["mu2wt"] - (p["wt2land"] + p["wt2rec"] + p["wt2back"])
        f["Waste treatment => Waste landfilled"][...] = p["wt2land"]
        f["Waste treatment => Backfilling"][...] = p["wt2back"]
        f["Waste treatment => Recycling"][...] = p["wt2rec"]

        f["Backfilling => Processed material"][...] = p["back2pm"]
        f["Recycling => Processed material"][...] = p["rec2pm"]

        f["Incineration => Total emissions"][...] = p["wt2te"]


        #MASS BALANCE DMI
        inflow_dmi = f["Imports => Direct material inputs"] + f["Natural resources extracted => Direct material inputs"]
        outflow_dmi = f["Direct material inputs => Processed material"]
        f["Direct material inputs => Residual imbalance"][...]  = inflow_dmi - outflow_dmi

        #MASS BALANCE PM
        inflow_pm = (f["Backfilling => Processed material"] + f["Recycling => Processed material"] + f["Direct material inputs => Processed material"])
        outflow_pm = (f["Processed material => Exports"] + f["Processed material => Dissipative flows"] + f["Processed material => Total emissions"] + f["Processed material => Material use"])
        f["Processed material => Residual imbalance"][...]  = inflow_pm - outflow_pm

        #MASS BALANCE TE
        inflow_te =  f["Processed material => Total emissions"] +  f["Incineration => Total emissions"]
        outflow_te =  f["Total emissions => Emissions to air"] + f["Total emissions => Emissions to water"]
        f["Total emissions => Residual imbalance"][...] = inflow_te - outflow_te

        #MASS BALANCE MU
        inflow_mu = f["Processed material => Material use"]
        outflow_mu = f["Material use => Waste treatment"] + f["Material use => Material accumulation"]
        f["Material use => Residual imbalance"][...] = inflow_mu - outflow_mu

        #MASS BALANCE WT
        inflow_wt = f["Material use => Waste treatment"]
        outflow_wt =  f["Waste treatment => Incineration"] + f["Waste treatment => Waste landfilled"] + f["Waste treatment => Backfilling"] + f["Waste treatment => Recycling"]
        f["Waste treatment => Residual imbalance"][...] = inflow_wt - outflow_wt

        #MASS BALANCE INC
        inflow_inc = f["Waste treatment => Incineration"]
        outflow_inc = f["Incineration => Total emissions"]
        f["Incineration => Residual imbalance"][...] = inflow_inc - outflow_inc

        #MASS BALANCE REC
        inflow_rec = f["Waste treatment => Recycling"]
        outflow_rec =  f["Recycling => Processed material"]
        f["Recycling => Residual imbalance"][...] = inflow_rec - outflow_rec

        #MASS BALANCE BACK
        inflow_back = f["Waste treatment => Backfilling"]
        outflow_back = f["Backfilling => Processed material"]
        f["Backfilling => Residual imbalance"][...] = inflow_back - outflow_back

        total_residual = (
            f["Direct material inputs => Residual imbalance"] 
            + f["Processed material => Residual imbalance"] 
            + f["Total emissions => Residual imbalance"] 
            + f["Material use => Residual imbalance"] 
            + f["Waste treatment => Residual imbalance"]
            + f["Recycling => Residual imbalance"]
            + f["Backfilling => Residual imbalance"]
            + f["Incineration => Residual imbalance"])
        
        f["Residual imbalance => sysenv"][...] = total_residual

        #Export flows
        f["Exports => sysenv"][...] = f["Processed material => Exports"]
        f["Dissipative flows => sysenv"][...] = f["Processed material => Dissipative flows"]
        f["Waste landfilled => sysenv"][...] = f["Waste treatment => Waste landfilled"]
        f["Emissions to air => sysenv"][...] = f["Total emissions => Emissions to air"]
        f["Emissions to water => sysenv"][...] = f["Total emissions => Emissions to water"]



### Step 9: Run our calculations 

Firstly we will initialize our MFA system. All the requirements for our mfa system is what we have been defining in the previous code cells. 

We therefore simply refer to their varaibles when initializing the system as seen below.

In [142]:
my_mfa_system = MyMFASystem(
    dims=dims,
    processes=processes,
    flows=flows,
    parameters=parameters,
    stocks = stocks,
)

my_mfa_system.compute()

for f in my_mfa_system.flows.values():
    print(f.name, "\n", f.to_df(), "\n")

sysenv => Imports 
                                              value
time Products                                     
2010 Biomass                          1.608982e+05
     Metal ores (gross ores)          2.280987e+05
     Non-metallic minerals            9.511935e+04
     Fossil energy materials_carrier  1.055930e+06
2011 Biomass                          1.684920e+05
...                                            ...
2059 Fossil energy materials_carrier  9.628419e+05
2060 Biomass                          2.932169e+05
     Metal ores (gross ores)          2.210998e+05
     Non-metallic minerals            1.806214e+05
     Fossil energy materials_carrier  9.629501e+05

[204 rows x 1 columns] 

sysenv => Natural resources extracted 
                                              value
time Products                                     
2010 Biomass                          1.457143e+06
     Metal ores (gross ores)          1.645491e+05
     Non-metallic minerals            3.117164e

### Step 9.2: Check mass balance

Here we may check for individual years to ensure our flow values aligns with our raw data values.

In [143]:
flow_names = [
'sysenv => Imports',
'sysenv => Natural resources extracted',
'Imports => Direct material inputs',
'Natural resources extracted => Direct material inputs',
'Direct material inputs => Processed material',
'Processed material => Exports',
'Processed material => Dissipative flows',
'Processed material => Total emissions',
'Processed material => Material use',
'Total emissions => Emissions to air',
'Total emissions => Emissions to water',
'Material use => Waste treatment',
'Material use => Material accumulation',
'Waste treatment => Incineration',
'Incineration => Total emissions',
'Waste treatment => Waste landfilled',
'Waste treatment => Backfilling',
'Backfilling => Processed material',
'Waste treatment => Recycling',
'Recycling => Processed material',
'Exports => sysenv',
'Dissipative flows => sysenv',
'Waste landfilled => sysenv',
'Emissions to air => sysenv',
'Emissions to water => sysenv',
'Residual imbalance => sysenv'
]

for flow_name in flow_names:
    values_2023 = my_mfa_system.flows[flow_name][{"t": 2023}]
    print("Values for", flow_name, "in 2023:", values_2023.values)

parameter_names = [    
    'nre', 
    'imports',
    'pm',
    'exports',
    'dis flow',
    'tem',
    'matuse',
    'mu2wt',
    'mu2ma',
    'wt2te',
    'te2air',
    'te2water',
    'wt2land',
    'wt2rec',
    'wt2back',
    'rec2pm',
    'back2pm'
]

for parameter_name in parameter_names:
    values_2023 = my_mfa_system.parameters[parameter_name][{"t": 2023}]
    print("Values for", parameter_name, "in 2023:", values_2023.values)

Values for sysenv => Imports in 2023: [192647.895 209848.069  94291.463 958944.343]
Values for sysenv => Natural resources extracted in 2023: [1498892.966  227804.549 3155488.652  341426.737]
Values for Imports => Direct material inputs in 2023: [192647.895 209848.069  94291.463 958944.343]
Values for Natural resources extracted => Direct material inputs in 2023: [1498892.966  227804.549 3155488.652  341426.737]
Values for Direct material inputs => Processed material in 2023: [1691541.  437653. 3249779. 1300371.]
Values for Processed material => Exports in 2023: [200807.603 115734.015  79598.731 223419.758]
Values for Processed material => Dissipative flows in 2023: [ 60377. 144905.  36226.      0.]
Values for Processed material => Total emissions in 2023: [1.106155e+06 0.000000e+00 6.100000e+01 1.046509e+06]
Values for Processed material => Material use in 2023: [ 474486.  265425. 3889276.   64072.]
Values for Total emissions => Emissions to air in 2023: [1163574.    5567.   11745. 10

Now we can finally conduct a mass balance check of our system using FLODYMs mass balance check command.

It is important to note that this command simply checks if:

$$ System \ Imports - System \ Exports = 0$$

As we have a stock accumulation in our system we do not check if this command returns true, instead we check that:
$$ Inflow - Outflow = \Delta Stock $$

In [144]:
balance_errors = my_mfa_system.check_mass_balance(raise_error=False)

We see that the difference in sysenv flows (ie. system imports - system exports) is equal to our change in stock. We can therefore confirm that **our system is in mass balance**.

### Step 10: Visualize MFA and Dashboard

We now visualize our MFA across all of our historic data, including circularit rates and stock accumulation.

We will utilize the dash and dash_bootstrap_componets packages to create a combined dashboard oveview of our system.

If you have not used these packages before, you will need to install by running the code below.

We are now ready to mkae our dynamic dashboard. Notice you can choose your timeline of interest with slide bar at the top.

In [150]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import dash_bootstrap_components as dbc
from plotly.colors import qualitative
from dash import Dash, html, dcc, Input, Output
import dash_bootstrap_components as dbc

app = Dash(__name__, external_stylesheets=[dbc.themes.COSMO])
server = app.server

# Extract model dimensions
years = np.array(my_mfa_system.dims["t"].items)
products = list(my_mfa_system.dims["p"].items)

from plotly.colors import qualitative

### --- Universal Color Palette ---
products = [str(p).strip() for p in products]
products = ['Biomass', 'Metal ores (gross ores)', 'Non-metallic minerals', 'Fossil energy materials_carrier']
palette = qualitative.Prism

MATERIAL_COLORS = {
    products[i]: palette[i % len(palette)]
    for i in range(len(products))
}

# --- Extract key values from MFA for Circularity and Stock ---
PM_in = my_mfa_system.flows["Direct material inputs => Processed material"].values + my_mfa_system.flows["Backfilling => Processed material"].values + my_mfa_system.flows["Recycling => Processed material"].values
REC_PM = my_mfa_system.flows["Recycling => Processed material"].values
ACC = my_mfa_system.flows["Material use => Material accumulation"].values

# DASH APP & LAYOUT

app = Dash(__name__, external_stylesheets=[dbc.themes.COSMO])
server = app.server

app.layout = dbc.Container(
    fluid=True,
    children=[
        dbc.Row([
            dbc.Col([
                html.H1("SS Earth for EU Member Countries",
                        className="text-center my-4", style={"fontWeight": 700})
            ])
        ]),

        dbc.Row([
            dbc.Col([
                html.Label("Select Year Range:", style={"fontWeight": 600}),
                dcc.RangeSlider(
                    id="year-range",
                    # Ensure min/max are integers if years is a NumPy array of integers
                    min=int(years.min()), 
                    max=int(years.max()),
                    step=1,
                    value=[2010, 2023],
                    # Use list comprehension for marks to handle NumPy array years
                    marks={int(y): str(int(y)) for y in years[::5]}, 
                    allowCross=False
                )
            ])
        ], className="mb-5"),

        # Sankey
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardHeader(html.H4("EU MFA Sankey Diagram")),
                    dbc.CardBody([
                        dcc.Graph(id="sankey-plot", style={"height": "600px"})
                    ])
                ])
            ])
        ], className="mb-5"),

        # Circularity + Accumulation
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardHeader(html.H4("Circularity Rate by Material")),
                    dbc.CardBody([
                        dcc.Graph(id="circularity-plot", style={"height": "450px"})
                    ])
                ])
            ], md=6),

            dbc.Col([
                dbc.Card([
                    dbc.CardHeader(html.H4("Annual Material Accumulation by Type")),
                    dbc.CardBody([
                        dcc.Graph(id="accumulation-plot", style={"height": "450px"})
                    ])
                ])
            ], md=6),
        ], className="mb-5"),

        # Aggregated Charts
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardHeader(html.H4("Aggregated Circularity Rate")),
                    dbc.CardBody([
                        dcc.Graph(id="agg-circ-plot", style={"height": "450px"})
                    ])
                ])
            ], md=6),

            dbc.Col([
                dbc.Card([
                    dbc.CardHeader(html.H4("Cumulative Material Accumulation by Material")),
                    dbc.CardBody([
                        dcc.Graph(id="agg-acc-plot", style={"height": "450px"})
                    ])
                ])
            ], md=6),
        ])
    ]
)


# --- CALLBACKS ---

# Sankey
@app.callback(
    Output("sankey-plot", "figure"),
    Input("year-range", "value")
)
def update_sankey(year_range):

    selected_year = year_range[1]  # max year

    colors = {"default": "gray"}

    colorful_flows = [
        f for f in my_mfa_system.flows.values()
        if "Products" in f.dims.names
    ]

    colors.update({
        f.name: ("Products", qualitative.Prism)
        for f in colorful_flows
    })
    
    #if "Residual imbalance => sysenv" in colors:
        #del colors["Residual imbalance => sysenv"]

    plotter = PlotlySankeyPlotter(
        mfa=my_mfa_system,
        split_flows_by="Products",
        slice_dict={"t": selected_year},
        flow_color_dict=colors
    )

    fig = plotter.plot()

    legend_traces = [
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            marker=dict(size=15, color=qualitative.Prism[i % len(qualitative.Prism)]),
            showlegend=True,
            name=product
    )
    for i, product in enumerate(products)
]

    for trace in legend_traces:
        fig.add_trace(trace)

    fig.update_layout(
        title=f"MFA Sankey Diagram – Year {selected_year} [Kt]",
        height=700
    )

    return fig


# ----- Circularity by Material -----
@app.callback(
    Output("circularity-plot", "figure"),
    Input("year-range", "value")
)
def update_circularity(year_range):

    ymin, ymax = year_range
    mask = (years >= ymin) & (years <= ymax)

    df = pd.DataFrame({"Year": years[mask]})

    for i, mat in enumerate(products):
        pm_vals = PM_in[:, i][mask]
        rec_vals = REC_PM[:, i][mask]
        # Ensure output array is float for division
        df[mat] = np.divide(rec_vals, pm_vals, out=np.zeros_like(rec_vals, dtype=float), where=pm_vals!=0)

    fig = px.line(df, x="Year", y=products,
                  labels={"value": "Circularity Rate [-]"},
                  color_discrete_map=MATERIAL_COLORS)
    fig.update_layout(yaxis_range=[0, 0.2],
                      legend_title_text='Material')

    return fig


# ----- Annual Accumulation -----
@app.callback(
    Output("accumulation-plot", "figure"),
    Input("year-range", "value")
)
def update_accumulation(year_range):

    ymin, ymax = year_range
    mask = (years >= ymin) & (years <= ymax)

    df = pd.DataFrame({"Year": years[mask]})
    
    # ACC is now a (T, P) NumPy array
    for i, mat in enumerate(products):
        df[mat] = ACC[:, i][mask] 

    fig = px.line(df, x="Year", y=products,
                  labels={"value": "Accumulation [Kt]"},
                  color_discrete_map=MATERIAL_COLORS)
    fig.update_layout(legend_title_text='Material')
    
    return fig


# ----- Aggregated Circularity -----
@app.callback(
    Output("agg-circ-plot", "figure"),
    Input("year-range", "value")
)
def update_agg_circularity(year_range):

    ymin, ymax = year_range
    mask = (years >= ymin) & (years <= ymax)

    total_pm = PM_in.sum(axis=1)[mask]
    total_rec = REC_PM.sum(axis=1)[mask]

    agg_circ = np.divide(total_rec, total_pm,
                         out=np.zeros_like(total_rec, dtype=float),
                         where=total_pm!=0)

    df = pd.DataFrame({
        "Year": years[mask],
        "Aggregated Circularity": agg_circ
    })

    fig = px.line(df, x="Year", y="Aggregated Circularity")
    fig.update_layout(yaxis_range=[min(agg_circ) - 0.002, max(agg_circ) + 0.002])
    return fig


# ----- Aggregated Stock (Cumulative per Material Type) -----
@app.callback(
    Output("agg-acc-plot", "figure"),
    Input("year-range", "value")
)
def update_agg_acc(year_range):
    
    ymin, ymax = year_range
    
    # Calculate the cumulative stock directly from the ACC (annual change) NumPy array
    cumulative_stock = np.cumsum(ACC, axis=0) 
    
    # Filter the years
    mask = (years >= ymin) & (years <= ymax)
    years_filtered = years[mask]
    stock_filtered = cumulative_stock[mask]

    # Create DataFrame for plotting
    df_data = {"Year": years_filtered}
    final_stocks = {} 

    for i, mat in enumerate(products):
        # The filtered stock array contains the cumulative stock up to each year in the range
        df_data[mat] = stock_filtered[:, i]
        
        # Calculate final stock for sorting purposes (use the value at the end of the selected range)
        final_stocks[mat] = stock_filtered[-1, i]

    final_df = pd.DataFrame(df_data)
    
    sorting_series = pd.Series(final_stocks).sort_values(ascending=True)
    sorted_materials = sorting_series.index.tolist()
    
    # Melt the DataFrame for Plotly Express (Wide to Long Format)
    df_melted = final_df.melt(
        id_vars=['Year'], 
        var_name='Material', 
        value_name='Cumulative Stock [Kt]'
    )

    fig = px.area(df_melted, 
        x="Year", 
        y="Cumulative Stock [Kt]",
        color="Material",
        labels={"Cumulative Stock [Kt]": "In-Use Stock [Kt]"},
        height=500,
        template='plotly_white',
        category_orders={"Material": sorted_materials}, # Enforces the custom stacking order
        color_discrete_map=MATERIAL_COLORS
    )
    
    fig.update_layout(
        margin={"t": 30, "b": 10, "l": 10, "r": 10},
        legend_title_text='Material'
    )
    return fig
import socket

def get_free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

server = get_free_port()

app.run(debug=True, port=server)


After running the dash app above you can open this link in your web browser for a full page view of the dashboard:

In [149]:
print(f'http://127.0.0.1:{server}/')

http://127.0.0.1:64763/


This assignment is designed for the DTU COURSE 12139 in Fall 2025 in implementing Prospective Model for Circularity and Management of SS Earth for EU in a Python environment.

This notebook has been created by group 57 during course F25 Resource Engineering 12139.

Part of this notebook was created on the basis of notebooks by Logan, H. (2025) from DTU Course 12139: Resource Engineering. Module 3: Prospective Model for Circularity and Management of SS Earth for EU.

Declaration of Generative AI: 

Sections of this notebook were drafted with the assistance of OpenAI’s ChatGPT model (accessed December 2025) and Google Gemini Pro (accessed December 2025) to support code development, technical formatting and visualization.

OpenAI, ChatGPT, December 2025, https://chat.openai.com/
Alphabet Inc, Google Gemini, December 2025, https://gemini.google.com/app